# 01 · MS MARCO full corpus: data + BM25 (CPU)

**Settings:** Accelerator **None** (CPU: uses no GPU quota) · Internet **On** · run with **Save Version → Save & Run All**.

Builds the 8.8M-passage collection, exports 200K training queries, builds the passage doc store,
and runs BM25 (the project's local BM25, score-for-score identical to Elasticsearch) on MS MARCO dev (6,980 queries),
TREC DL 2019 and 2020. Everything needed later is written to `/kaggle/working` (≈ 4 GB).

Estimated runtime: 1–1.5 h.

In [ ]:
# Clone the project (outside /kaggle/working: code is not an output) and install it.
import os, subprocess, sys, time, json, glob, shutil
T0 = time.time()
REPO = os.environ.get("REPO", "https://github.com/Rishabh763/Search-engine-Neural-Ranking.git")
import socket
try:
    socket.gethostbyname("github.com")
except OSError:
    raise RuntimeError(
        "No internet in this notebook. In the right sidebar open Session options "
        "(or Settings) and switch Internet ON, then restart and run again. "
        "The toggle only appears once your Kaggle account is phone-verified "
        "(Settings > Phone verification).") from None
if not os.path.exists("/tmp/repo"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/tmp/repo"], check=True)
if not os.path.exists("/tmp/repo/scripts/train_reranker.py"):
    raise RuntimeError(
        "The repo on GitHub is older than this notebook (no scripts/train_reranker.py). "
        "Commit and push from your laptop, then delete /tmp/repo (!rm -rf /tmp/repo) and rerun.")
os.chdir("/tmp/repo")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
!pip install -q -e . 2>&1 | tail -3

In [ ]:
os.environ.update(
    DATA_DIR="/kaggle/working/data/processed",
    INDEX_DIR="/kaggle/working/indexes",
    RUNS_DIR="/kaggle/working/runs",
    IR_DATASETS_HOME="/tmp/ir_datasets",   # raw downloads: not kept as output
    PYTHONUTF8="1",
)
!python -c "from searchengine.envinfo import environment_info, short_label; print(short_label(environment_info()))"

## 1. Full collection (downloads ~1 GB, writes 8.8M passages)

In [ ]:
!python scripts/prepare_data.py --name msmarco-full --corpus msmarco-passage --eval-sets msmarco-dev dl19 dl20 2>&1 | tail -25

In [ ]:
print(f"elapsed: {(time.time() - T0) / 60:.1f} min"); print(subprocess.run(["du", "-sh", "/kaggle/working"], capture_output=True, text=True).stdout)

## 2. Training queries (200K of the ~500K judged MS MARCO train queries; disjoint from all eval sets)

In [ ]:
!python scripts/prepare_train.py --collection msmarco-full --n-queries 200000

## 3. Passage doc store (byte offsets for random access by id)

In [ ]:
!python -c "from searchengine.config import get_settings; from searchengine.data.collection import Collection; from searchengine.docstore import DocStore; s=get_settings(); d=DocStore.open(Collection.from_name('msmarco-full', s.data_dir), s.index_dir); print(len(d), 'passages indexed')"

## 4. BM25 over 8.8M passages (local implementation, Lucene-identical scoring)

In [ ]:
!python scripts/build_index.py --collection msmarco-full --backend local 2>&1 | tail -8

In [ ]:
print(f"elapsed: {(time.time() - T0) / 60:.1f} min"); print(subprocess.run(["du", "-sh", "/kaggle/working"], capture_output=True, text=True).stdout)

In [ ]:
!python scripts/evaluate.py --collection msmarco-full --retriever bm25-local --latency-queries 0

Sanity anchors on the full corpus (Anserini BM25, k1=0.9 b=0.4): dev MRR@10 ≈ 0.184, DL19 nDCG@10 ≈ 0.506, DL20 ≈ 0.480.
Ours should land within ~0.01 (same scoring; tokenization matches Lucene on >99% of passages).

In [ ]:
# The BM25 index itself (~2-3 GB) isn't needed downstream (its runs are saved), so drop it from the output.
shutil.rmtree("/kaggle/working/indexes/msmarco-full/bm25-local", ignore_errors=True)
for f in sorted(glob.glob("/kaggle/working/runs/msmarco-full/*/bm25-local.json")):
    r = json.load(open(f)); print(f"{r['eval_set']:12s} n={r['n_queries']:5d}  " + "  ".join(f"{k}={v:.4f}" for k, v in r["metrics"].items()))

In [ ]:
print(f"elapsed: {(time.time() - T0) / 60:.1f} min"); print(subprocess.run(["du", "-sh", "/kaggle/working"], capture_output=True, text=True).stdout)